In [1]:
!pip install -q -U opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-grpc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 735.2 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 1.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 5.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/models/doaa87/l/other/default/1/memory.py
/kaggle/input/models/doaa87/l/other/default/1/config.py


In [3]:
!pip install -q langchain-community langchain-text-splitters pypdf requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.2 MB/s eta 0:00:00


In [4]:
"""Step 1: load the PDF and split it into chunks."""
from pathlib import Path

import requests
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Settings (kept here so this file works on its own)
DATA_DIR = Path("data")
CHUNK_SIZE = 800
CHUNK_OVERLAP = 120

# Stanford CS329S slides: Data Distribution Shifts on Streaming Data
PDF_URL = (
    "https://stanford-cs329s.github.io/slides/"
    "cs329s_11_slides_data_distribution_shifts_streams_shreya_shankar.pdf"
)
PDF_PATH = DATA_DIR / "data_distribution_shifts.pdf"


def download_pdf(url: str = PDF_URL, path: Path = PDF_PATH) -> Path:
    """Download the PDF once; skip the download if the file already exists."""
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        path.write_bytes(r.content)
    return path


def load_and_split(pdf_path: Path = PDF_PATH):
    """Read the PDF page by page, split it into chunks, and keep the page number in each chunk's metadata."""
    pages = PyPDFLoader(str(pdf_path)).load()  # one Document per page

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
    )
    chunks = splitter.split_documents(pages)

    for c in chunks:
        c.metadata["page"] += 1  # PyPDF counts pages from 0; we want 1-based page numbers
        c.metadata["source"] = Path(c.metadata["source"]).name  # file name only, not the full path
    return chunks


if __name__ == "__main__":
    chunks = load_and_split(download_pdf())
    print(f"Number of chunks: {len(chunks)}")
    print("First chunk metadata:", chunks[0].metadata)
    print(chunks[0].page_content[:300])

/tmp/ipykernel_49/2752571390.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
Ignoring wrong pointing object 8 0 (offset 0)


Number of chunks: 36
First chunk metadata: {'producer': 'macOS Version 12.0 (Build 21A344) Quartz PDFContext', 'creator': 'Keynote', 'creationdate': "D:20220209230246Z00'00'", 'title': 'feb9', 'moddate': "D:20220209230246Z00'00'", 'source': 'data_distribution_shifts.pdf', 'total_pages': 36, 'page': 1, 'page_label': '1'}
Shreya Shankar, UC Berkeley 
🐻 
February 2022
Towards Observability for 
Machine Learning Pipelines
Monitoring Streaming ML with Feedback Delays
1


In [5]:
!pip install -q langchain-chroma langchain-huggingface sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 608.4 kB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 78.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 110.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 78.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.2 MB/s eta 0:00:00


In [6]:
# Step 2: dense (semantic) search with embeddings + ChromaDB
# Needs `chunks` from the Step 1 cell
import os
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # small and fast
CHROMA_DIR = "chroma_db"  # folder where the vectors are saved
TOP_K = 5

# The model that turns text into vectors (numbers that capture meaning)
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

# Load Chroma if it already exists, otherwise build it from the chunks.
# This avoids storing the same chunks twice when you re-run the cell.
if os.path.exists(CHROMA_DIR):
    vectordb = Chroma(persist_directory=CHROMA_DIR, embedding_function=embeddings)
else:
    vectordb = Chroma.from_documents(
        chunks, embedding=embeddings, persist_directory=CHROMA_DIR
    )


def dense_search(query: str, k: int = TOP_K):
    """Return the k chunks whose meaning is closest to the query."""
    return vectordb.similarity_search(query, k=k)


# Quick test
for doc in dense_search("What is data distribution shift?"):
    text = doc.page_content[:150].replace("\n", " ")
    print(f"[page {doc.metadata['page']}] {text}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[page 10] Shift Recap Notation  🎶 •X is feature (covariate) space, Y is label space  •P(X): distribution of features  •P(Y): distribution of labels  •P(X | Y): 
[page 16] Shift Recap Examples  🧑🏫 •X = features (e.g., location), Y = labels (high tip indicator)   •Covariate shift  •More taxi rides in Midtown area around N
[page 11] Shift Recap Terminology  🧑🏫 •Covariate shift  •P(Y | X) is the same but P(X) changes  •Label shift  •P(Y) changes but P(X | Y) is the same  •Concept s
[page 12] Existing Methods for Tackling Shift Levels of sophistication  👸 •Straw-man approach  🥤  •Tracking means & quantiles of features and outputs  •“I took 
[page 32] Fine-grained Monitoring  🔎 Diagnosing performance issues: towards retraining models  🔁 •Using existing methods to compute shift doesn’t work in practi


In [7]:
!pip install -q rank-bm25

In [8]:
# Step 3: hybrid search = BM25 (keywords) + dense (meaning), merged with RRF
# Needs `chunks` (Step 1) and `dense_search` (Step 2)
import re
from rank_bm25 import BM25Okapi

TOP_K_BM25 = 10
TOP_K_DENSE = 10
RRF_K = 60  # standard constant for RRF
TOP_K_FUSED = 10


def tokenize(text: str):
    """Lowercase and split into words."""
    return re.findall(r"\w+", text.lower())


# Build the BM25 index on the same chunks that are in Chroma
bm25 = BM25Okapi([tokenize(c.page_content) for c in chunks])


def bm25_search(query: str, k: int = TOP_K_BM25):
    """Return the top k chunks by keyword match (skip chunks with score 0)."""
    scores = bm25.get_scores(tokenize(query))
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [chunks[i] for i in top_idx if scores[i] > 0]


def doc_key(doc):
    """Identify a chunk so the same chunk from both searches is counted as one."""
    return (doc.metadata["page"], doc.page_content)


def rrf_fuse(result_lists, k: int = TOP_K_FUSED):
    """Merge several ranked lists. Each chunk gets 1 / (RRF_K + rank) from every list it appears in."""
    scores, docs = {}, {}
    for results in result_lists:
        for rank, doc in enumerate(results, start=1):
            key = doc_key(doc)
            docs[key] = doc
            scores[key] = scores.get(key, 0) + 1 / (RRF_K + rank)
    best = sorted(scores, key=scores.get, reverse=True)[:k]
    return [docs[key] for key in best]


def hybrid_search(query: str, k: int = TOP_K_FUSED):
    """Run dense and BM25 search, then merge the results with RRF."""
    return rrf_fuse(
        [dense_search(query, TOP_K_DENSE), bm25_search(query, TOP_K_BM25)], k=k
    )


# Quick test
for doc in hybrid_search("What is covariate shift?")[:5]:
    text = doc.page_content[:150].replace("\n", " ")
    print(f"[page {doc.metadata['page']}] {text}")

[page 11] Shift Recap Terminology  🧑🏫 •Covariate shift  •P(Y | X) is the same but P(X) changes  •Label shift  •P(Y) changes but P(X | Y) is the same  •Concept s
[page 10] Shift Recap Notation  🎶 •X is feature (covariate) space, Y is label space  •P(X): distribution of features  •P(Y): distribution of labels  •P(X | Y): 
[page 16] Shift Recap Examples  🧑🏫 •X = features (e.g., location), Y = labels (high tip indicator)   •Covariate shift  •More taxi rides in Midtown area around N
[page 33] Fine-grained Monitoring  🔎 Diagnosing performance issues: towards retraining models  🔁 •Diﬀerent shifts imply diﬀerent retraining strategies, e.g.,  •
[page 9] Why is this Hard? Data “shifts”…  🤔 •Determining real-time performance requires labels  •…which are not always available post-deployment  •Is performa


In [9]:
# Step 4: rerank the hybrid results with a cross-encoder
# Needs `hybrid_search` from Step 3
from sentence_transformers import CrossEncoder

RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
TOP_K_FINAL = 4

# A model that reads the question and a chunk together and gives them a relevance score
reranker = CrossEncoder(RERANK_MODEL)


def rerank(query: str, docs, k: int = TOP_K_FINAL):
    """Score every (query, chunk) pair and return the k best chunks."""
    pairs = [(query, d.page_content) for d in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ranked[:k]]


def retrieve(query: str):
    """Full retrieval pipeline: hybrid search, then rerank."""
    return rerank(query, hybrid_search(query))


# Quick test
for doc in retrieve("What is covariate shift?"):
    text = doc.page_content[:150].replace("\n", " ")
    print(f"[page {doc.metadata['page']}] {text}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

[page 11] Shift Recap Terminology  🧑🏫 •Covariate shift  •P(Y | X) is the same but P(X) changes  •Label shift  •P(Y) changes but P(X | Y) is the same  •Concept s
[page 33] Fine-grained Monitoring  🔎 Diagnosing performance issues: towards retraining models  🔁 •Diﬀerent shifts imply diﬀerent retraining strategies, e.g.,  •
[page 10] Shift Recap Notation  🎶 •X is feature (covariate) space, Y is label space  •P(X): distribution of features  •P(Y): distribution of labels  •P(X | Y): 
[page 16] Shift Recap Examples  🧑🏫 •X = features (e.g., location), Y = labels (high tip indicator)   •Covariate shift  •More taxi rides in Midtown area around N


In [10]:
!pip install -q -U "bitsandbytes>=0.46.1" accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 18.2 MB/s eta 0:00:00


In [11]:
# Step 5a: load Qwen2.5-7B-Instruct in 4-bit so it fits on a Kaggle GPU
# !pip install -q bitsandbytes accelerate   (run once if they are missing)
import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline,
)
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL, quantization_config=bnb_config, device_map="auto"
)

# Use max_new_tokens instead of the model's default max_length
model.generation_config.max_length = None

# Make the answers stable: no random sampling
# Back to sampling with a low temperature
# Greedy decoding: same question gives the same answer, with fewer random slips
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False,
    return_full_text=False,
)
chat_llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=pipe))

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [12]:
# Step 5 (final): answer ANY English question about ANY PDF
# Needs: `chunks` (Step 1), `retrieve` + `reranker` (Step 4), `chat_llm` (Step 5a)
import re

from IPython.display import Markdown, display
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ---------------- Prompts ----------------
# 1) Router: is the question about the whole PDF or about one thing?
router_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Classify a question about a document. Answer with ONE word.\n"
     "OVERVIEW: the question asks what the whole document covers "
     "(its topics, outline, table of contents or summary).\n"
     "SPECIFIC: the question asks about one particular topic, term or fact."),
    ("human", "{question}"),
])

# 2) Answer: only from the retrieved context
explain_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a friendly tutor. Use ONLY the context below. "
     "Do not add outside knowledge and do not invent scenarios or numbers. "
     "If the context does not contain the answer, say you don't know. "
     "Answer the question (or explain the topic) in 3-4 sentences, "
     "reusing the wording of the context. Reply with the answer only."),
    ("human", "Context:\n{context}\n\nQuestion or topic: {question}"),
])

# 3) Example picker: the model only chooses a NUMBER, the code returns the real line
pick_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Below are numbered lines taken from a PDF. Each line may start with its heading "
     "(the words before the first colon). Pick the ONE line that is the best "
     "concrete example for the question, and its heading should match the topic. "
     "The example must describe a real-world situation, not a definition or a formula. "
     "Answer with the number only. If no line is an example, answer 0."),
    ("human", "Question: {question}\n\nLines:\n{lines}"),
])

router_chain = router_prompt | chat_llm | StrOutputParser()
explain_chain = explain_prompt | chat_llm | StrOutputParser()
pick_chain = pick_prompt | chat_llm | StrOutputParser()


# ---------------- Helpers (nothing here is specific to one PDF) ----------------
def format_context(docs):
    """Put the page number before each chunk so the model can see it."""
    return "\n\n".join(f"[page {d.metadata['page']}]\n{d.page_content}" for d in docs)


def slide_title(text: str) -> str:
    """Title = the text before the first bullet or emoji (or the first 60 characters)."""
    flat = " ".join(text.split())
    return re.split(r"[•\U0001F300-\U0001FAFF]", flat)[0].strip()[:60]


def candidate_lines(docs, max_lines: int = 25):
    """Cut the retrieved chunks into short lines: a list of (page, line).
    Each line keeps its heading (e.g. "Label shift") so the model knows what it is about."""
    lines, seen = [], set()
    for d in docs:
        flat = " ".join(d.page_content.split())  # collapse line breaks
        flat = re.sub(r"[\U0001F300-\U0001FAFF\u200d]", "", flat)  # remove emojis
        heading = ""
        for piece in re.split(r"•|(?<=[.!?])\s+", flat):
            piece = re.sub(r"(\s+\d{1,3})+$", "", piece.strip())  # drop a slide number at the end
            n_words = len(piece.split())
            if not re.search(r"[A-Za-z]", piece):
                continue  # skip pieces without letters
            if n_words <= 3:  # a short piece is a heading
                heading = piece
            elif n_words <= 30:
                line = f"{heading}: {piece}" if heading else piece
                if line not in seen:
                    seen.add(line)
                    lines.append((d.metadata["page"], line))
    return lines[:max_lines]


def best_page(text: str, docs) -> int:
    """Find the retrieved chunk that matches this sentence best (cross-encoder from Step 4)."""
    scores = reranker.predict([(text, d.page_content) for d in docs])
    return docs[int(scores.argmax())].metadata["page"]


def to_sentences(paragraph: str):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", paragraph) if s.strip()]


def pick_example(question: str, docs):
    """The model picks a line NUMBER; we return the real (page, line), or None."""
    lines = candidate_lines(docs)
    numbered = "\n".join(f"{i}. {line}" for i, (page, line) in enumerate(lines, 1))
    reply = pick_chain.invoke({"question": question, "lines": numbered})
    m = re.search(r"\d+", reply)
    n = int(m.group()) if m else 0
    return lines[n - 1] if 1 <= n <= len(lines) else None


# The outline is built by the code, not by the model: one title per page
outline = {}
for c in chunks:  # chunks are in page order, so the first chunk of a page gives its title
    title = slide_title(c.page_content)
    if title:
        outline.setdefault(c.metadata["page"], title)


def show_outline():
    lines = ["## 🗂️ Topics covered in this PDF", ""]
    lines += [f"- **Page {p}** — {t}" for p, t in sorted(outline.items())]
    display(Markdown("\n".join(lines)))


def build_markdown(question: str, answer: str, example, docs) -> str:
    answer = re.sub(r"^\s*(answer|explanation)\s*:\s*", "", answer.replace("**", "").strip(), flags=re.I)
    out = [f"## 📘 {question}", "", "### 💡 Answer"]

    if "don't know" in answer.lower():  # the PDF does not contain the answer
        out.append(f"- {answer}")
        return "\n".join(out)

    for s in to_sentences(answer):
        out.append(f"- {s} *(best match: page {best_page(s, docs)})*")

    if example:  # a real line from the PDF, with its real page
        page, line = example
        out += ["", "### 🧪 Example (a line from the PDF)", f"- “{line}” **[page {page}]**"]

    # Sources come from the code (not from the model), one line per page
    titles = {}
    for d in docs:
        titles.setdefault(d.metadata["page"], slide_title(d.page_content))
    out += ["", "### 📚 Sources"]
    out += [f"- **Page {p}** — {titles[p]}" for p in sorted(titles)]
    return "\n".join(out)


# ---------------- The one function you call ----------------
def ask(question: str):
    kind = router_chain.invoke({"question": question})
    if "OVERVIEW" in kind.upper():
        show_outline()
        return

    docs = retrieve(question)
    answer = explain_chain.invoke({"context": format_context(docs), "question": question})
    example = None if "don't know" in answer.lower() else pick_example(question, docs)
    display(Markdown(build_markdown(question, answer, example, docs)))


# Quick test
ask("What topics does this PDF cover?")
ask("Why is monitoring data shift hard?")
ask("What is gradient boosting?")

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


## 🗂️ Topics covered in this PDF

- **Page 1** — Shreya Shankar, UC Berkeley
- **Page 2** — Agenda
- **Page 3** — Dealing with ML Pipelines Sucks
- **Page 4** — Production ML An on-call engineer’s biggest nightmare
- **Page 5** — Production ML An on-call engineer’s biggest nightmare
- **Page 6** — Why Observability?
- **Page 7** — Types of ML Data Management Solutions 7 Pre-training
- **Page 8** — Real-Time ML Performance Monitoring: Background
- **Page 9** — Why is this Hard? Data “shifts”…
- **Page 10** — Shift Recap Notation
- **Page 11** — Shift Recap Terminology
- **Page 12** — Existing Methods for Tackling Shift Levels of sophistication
- **Page 13** — Toy ML Task: Running Example
- **Page 14** — Task familiarization
- **Page 15** — Pipeline familiarization
- **Page 16** — Shift Recap Examples
- **Page 17** — Real-Time ML Performance Monitoring: Challenges
- **Page 18** — Challenge Tree
- **Page 19** — Coarse-grained Monitoring
- **Page 20** — Coarse-grained Monitoring
- **Page 21** — Coarse-grained Monitoring
- **Page 22** — Coarse-grained Monitoring
- **Page 23** — Coarse-grained Monitoring
- **Page 24** — Coarse-grained Monitoring
- **Page 25** — Coarse-grained Monitoring
- **Page 26** — Coarse-grained Monitoring
- **Page 27** — Coarse-grained Monitoring
- **Page 28** — Coarse-grained Monitoring
- **Page 29** — Coarse-grained Monitoring
- **Page 30** — Coarse-grained Monitoring
- **Page 31** — Fine-grained Monitoring
- **Page 32** — Fine-grained Monitoring
- **Page 33** — Fine-grained Monitoring
- **Page 34** — mltrace: Ongoing Work ⚙ 34
- **Page 35** — Ongoing Research Projects
- **Page 36** — Readings and Resources

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## 📘 Why is monitoring data shift hard?

### 💡 Answer
- Monitoring data shift is hard because determining real-time performance requires labels that are not always available post-deployment, and degenerate feedback loops can occur when predictions influence feedback used to extract labels. *(best match: page 9)*
- Additionally, existing methods like the KS test may not work in practice due to low p-values for small datasets, leading to alert fatigue. *(best match: page 32)*

### 🧪 Example (a line from the PDF)
- “Using existing methods to compute shift doesn’t work in practice” **[page 32]**

### 📚 Sources
- **Page 9** — Why is this Hard? Data “shifts”…
- **Page 18** — Challenge Tree
- **Page 31** — Fine-grained Monitoring
- **Page 32** — Fine-grained Monitoring

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## 📘 What is gradient boosting?

### 💡 Answer
- The context provided does not contain information about gradient boosting. *(best match: page 9)*

### 📚 Sources
- **Page 9** — Why is this Hard? Data “shifts”…
- **Page 23** — Coarse-grained Monitoring
- **Page 24** — Coarse-grained Monitoring
- **Page 33** — Fine-grained Monitoring

In [13]:
ask("What is Covariate  shift")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## 📘 What is Covariate  shift

### 💡 Answer
- Covariate shift occurs when the distribution of features \( P(X) \) changes, but the relationship between features and labels \( P(Y | X) \) remains the same. *(best match: page 10)*

### 🧪 Example (a line from the PDF)
- “Covariate shift: More taxi rides in Midtown area around NYE” **[page 16]**

### 📚 Sources
- **Page 10** — Shift Recap Notation
- **Page 11** — Shift Recap Terminology
- **Page 16** — Shift Recap Examples
- **Page 33** — Fine-grained Monitoring

In [14]:
ask("What is Concept shift")

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## 📘 What is Concept shift

### 💡 Answer
- Concept shift is when the relationship between features and labels changes, but the distribution of features remains the same. *(best match: page 11)*
- For example, heavy construction in certain areas causing people to tip less, where the features (locations) do not change, but the tipping behavior given those locations does. *(best match: page 16)*

### 🧪 Example (a line from the PDF)
- “Concept shift: Heavy construction in certain areas causes people to tip less” **[page 16]**

### 📚 Sources
- **Page 10** — Shift Recap Notation
- **Page 11** — Shift Recap Terminology
- **Page 16** — Shift Recap Examples
- **Page 33** — Fine-grained Monitoring

In [15]:
ask("What is Label shift")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## 📘 What is Label shift

### 💡 Answer
- Label shift occurs when the distribution of labels \( P(Y) \) changes but the distribution of features given specific labels \( P(X | Y) \) remains the same. *(best match: page 10)*
- For example, if stimulus checks cause people to tip more, the tipping behavior distribution changes (\( P(Y) \) changes), but the relationship between features like location and tipping behavior stays the same (\( P(X | Y) \) is unchanged). *(best match: page 16)*

### 🧪 Example (a line from the PDF)
- “Label shift: Stimulus check causes people to tip more” **[page 16]**

### 📚 Sources
- **Page 9** — Why is this Hard? Data “shifts”…
- **Page 10** — Shift Recap Notation
- **Page 11** — Shift Recap Terminology
- **Page 16** — Shift Recap Examples

In [16]:
ask ('Why Observability?')

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## 📘 Why Observability?

### 💡 Answer
- Why Observability? *(best match: page 6)*
- We want to help engineers detect and diagnose bugs while supporting a wide variety of skill sets, minimizing downtime even if we can't catch all bugs before they happen. *(best match: page 6)*

### 🧪 Example (a line from the PDF)
- “Why Observability?: Can’t catch all bugs before they happen, but we want to minimize downtime” **[page 6]**

### 📚 Sources
- **Page 1** — Shreya Shankar, UC Berkeley
- **Page 6** — Why Observability?
- **Page 35** — Ongoing Research Projects
- **Page 36** — Readings and Resources

In [17]:
# ===== Step 6: Quiz + Grading with structured output =====
from typing import List
from pydantic import BaseModel, Field, field_validator
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.exceptions import OutputParserException
from langchain_core.prompts import ChatPromptTemplate

# ---------- 1) Schemas ----------
class QuizQuestion(BaseModel):
    question: str
    options: List[str] = Field(description="exactly 4 answer options")
    answer_index: int = Field(description="index (0-3) of the correct option")
    page: int = Field(default=0, description="page number (code overwrites this)")

    @field_validator("options")
    @classmethod
    def four_options(cls, v):
        if len(v) != 4:
            raise ValueError("need exactly 4 options")
        return v

    @field_validator("answer_index")
    @classmethod
    def valid_index(cls, v):
        if not 0 <= v <= 3:
            raise ValueError("answer_index must be 0-3")
        return v

class Quiz(BaseModel):
    topic: str
    questions: List[QuizQuestion]

class Evaluation(BaseModel):
    score: int = Field(ge=0, le=100, description="0-100")
    feedback: str
    correct_answer: str

quiz_parser = PydanticOutputParser(pydantic_object=Quiz)
eval_parser = PydanticOutputParser(pydantic_object=Evaluation)

# ---------- 2) Prompts ----------
# {hint} is empty on the first try and filled on retries,
# because greedy decoding would otherwise return the same bad output again.
quiz_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a tutor. Write multiple-choice questions using ONLY the context. "
     "Each question has exactly 4 options and one correct answer. "
     "Set page to 0. Return ONLY valid JSON, no extra text.\n{format_instructions}"),
    ("human",
     "Context:\n{context}\n\nTopic: {topic}\nWrite {n} questions.{hint}"),
]).partial(format_instructions=quiz_parser.get_format_instructions())

eval_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a strict grader. Use ONLY the context to decide the correct answer. "
     "If the student's answer contradicts the context, give a low score (0-30). "
     "If it is fully correct, give 90-100. Return ONLY valid JSON, no extra text.\n"
     "{format_instructions}"),
    ("human",
     "Context:\n{context}\n\nQuestion: {question}\nStudent answer: {answer}{hint}"),
]).partial(format_instructions=eval_parser.get_format_instructions())

quiz_chain = quiz_prompt | chat_llm | quiz_parser
eval_chain = eval_prompt | chat_llm | eval_parser

# ---------- 3) Retry helper ----------
def run_with_retry(chain, inputs, tries=3):
    hint = ""
    for i in range(tries):
        try:
            return chain.invoke({**inputs, "hint": hint})
        except OutputParserException as e:
            print(f"[retry {i+1}/{tries}] bad output, trying again")
            hint = "\n\nIMPORTANT: your last answer was invalid. Return ONLY valid JSON that matches the format."
    raise RuntimeError("Model failed to return valid JSON after retries")

# ---------- 4) Page lookup done by code (not by the model) ----------
def page_of(text, docs):
    # pick the retrieved chunk most similar to the text (estimate, not proof)
    scores = reranker.predict([(text, d.page_content) for d in docs])
    return docs[int(scores.argmax())].metadata["page"]

# ---------- 5) Main functions ----------
def generate_quiz(topic, n=3):
    docs = retrieve(topic)
    context = format_context(docs)
    quiz = run_with_retry(quiz_chain, {"context": context, "topic": topic, "n": n})
    for q in quiz.questions:
        correct = q.options[q.answer_index]
        q.page = page_of(q.question + " " + correct, docs)  # overwrite model page
    return quiz

def evaluate_answer(question, answer):
    docs = retrieve(question)
    context = format_context(docs)
    return run_with_retry(eval_chain,
                          {"context": context, "question": question, "answer": answer})

# ---------- 6) Test ----------
quiz = generate_quiz("data distribution shift", n=3)
for i, q in enumerate(quiz.questions):
    print(f"Q{i+1}: {q.question}")
    for j, o in enumerate(q.options):
        print(f"   {j}) {o}")
    print(f"   correct = {q.answer_index} | best match: page {q.page}\n")

q = quiz.questions[0]
right = q.options[q.answer_index]
wrong = next(o for j, o in enumerate(q.options) if j != q.answer_index)

print("--- RIGHT answer ---")
print(evaluate_answer(q.question, right))
print("--- WRONG answer ---")
print(evaluate_answer(q.question, wrong))

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[retry 1/3] bad output, trying again


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q1: What does P(X | Y) represent in the context of machine learning?
   0) Distribution of features given specific labels
   1) Distribution of labels given specific features
   2) Distribution of features
   3) Distribution of labels
   correct = 0 | best match: page 10

Q2: Which type of monitoring focuses on detecting performance issues with label delays?
   0) Fine-grained monitoring
   1) Coarse-grained monitoring
   2) Full-feedback monitoring
   3) Partial-feedback monitoring
   correct = 1 | best match: page 18

Q3: What is a challenge in fine-grained monitoring when dealing with thousands of features?
   0) Computing shift using existing methods works well
   1) Low alert fatigue
   2) Tedious scaling to hundreds of features
   3) No need for DQ verification
   correct = 2 | best match: page 31

--- RIGHT answer ---


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


score=100 feedback="The student's answer accurately describes what P(X | Y) represents in the context of machine learning." correct_answer='Distribution of features given specific labels'
--- WRONG answer ---
score=30 feedback="The student's answer contradicts the context provided. P(X | Y) represents the distribution of features given specific labels, not the distribution of labels given specific features." correct_answer='Distribution of features given specific labels'


In [18]:
# ===== Step 7a: Score history (memory) =====
import json, os, time
from collections import defaultdict

SCORES_FILE = "/kaggle/working/scores.json"  # may be lost if the session ends: download it

def load_history():
    if not os.path.exists(SCORES_FILE):
        return []
    with open(SCORES_FILE) as f:
        return json.load(f)

def save_score(topic, question, score):
    history = load_history()
    history.append({"topic": topic, "question": question,
                    "score": score, "time": time.strftime("%Y-%m-%d %H:%M")})
    with open(SCORES_FILE, "w") as f:
        json.dump(history, f, indent=2)

def average_score(topic=None):
    items = [h["score"] for h in load_history() if topic is None or h["topic"] == topic]
    return sum(items) / len(items) if items else None

def weakest_topic():
    # topic with the lowest average score (None if no history yet)
    by_topic = defaultdict(list)
    for h in load_history():
        by_topic[h["topic"]].append(h["score"])
    if not by_topic:
        return None
    return min(by_topic, key=lambda t: sum(by_topic[t]) / len(by_topic[t]))

# ---------- Test with the two results from Step 6 ----------
save_score("data distribution shift", q.question, 100)
save_score("monitoring", "test question", 30)

print("history size:", len(load_history()))
print("overall avg:", average_score())
print("weakest topic:", weakest_topic())  # expected: monitoring
# os.remove(SCORES_FILE)  # uncomment to clear the test data

history size: 2
overall avg: 65.0
weakest topic: monitoring


In [19]:
# ===== Step 7b: FastAPI backend (runs inside the notebook, same process as the model) =====
import io, threading, time, requests, uvicorn
from contextlib import redirect_stdout
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()
gpu_lock = threading.Lock()  # one model call at a time

class AskReq(BaseModel):
    question: str

class QuizReq(BaseModel):
    topic: str
    n: int = 3

class GradeReq(BaseModel):
    topic: str
    question: str
    answer: str

@app.post("/ask")
def api_ask(req: AskReq):
    with gpu_lock:
        buf = io.StringIO()
        with redirect_stdout(buf):      # ask() prints its result, so we capture the text
            ret = ask(req.question)
    text = ret if isinstance(ret, str) else buf.getvalue()
    return {"answer": text}

@app.post("/quiz")
def api_quiz(req: QuizReq):
    with gpu_lock:
        quiz = generate_quiz(req.topic, req.n)
    return quiz.model_dump()

@app.post("/grade")
def api_grade(req: GradeReq):
    with gpu_lock:
        ev = evaluate_answer(req.question, req.answer)
    save_score(req.topic, req.question, ev.score)
    return ev.model_dump()

@app.get("/history")
def api_history():
    return {"history": load_history(),
            "average": average_score(),
            "weakest_topic": weakest_topic()}

# ---------- start the server in a background thread ----------
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)

# ---------- quick test ----------
print(requests.get("http://127.0.0.1:8000/history").json())
r = requests.post("http://127.0.0.1:8000/ask", json={"question": "What is covariate shift?"})
print(r.json()["answer"])

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'history': [{'topic': 'data distribution shift', 'question': 'What does P(X | Y) represent in the context of machine learning?', 'score': 100, 'time': '2026-10-11 04:23'}, {'topic': 'monitoring', 'question': 'test question', 'score': 30, 'time': '2026-10-11 04:23'}], 'average': 65.0, 'weakest_topic': 'monitoring'}


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## 📘 What is covariate shift?

### 💡 Answer
- Covariate shift occurs when the distribution of features \( P(X) \) changes, but the relationship between features and labels \( P(Y | X) \) remains the same. *(best match: page 10)*

### 🧪 Example (a line from the PDF)
- “Covariate shift: More taxi rides in Midtown area around NYE” **[page 16]**

### 📚 Sources
- **Page 10** — Shift Recap Notation
- **Page 11** — Shift Recap Terminology
- **Page 16** — Shift Recap Examples
- **Page 33** — Fine-grained Monitoring

In [20]:
import os
if os.path.exists(SCORES_FILE):
    os.remove(SCORES_FILE)
print("history cleared:", load_history())  # expected: []

history cleared: []


In [21]:
# ===== Test: /quiz and /grade through the API =====
BASE = "http://127.0.0.1:8000"

# 1) Ask the API for a quiz
quiz = requests.post(f"{BASE}/quiz", json={"topic": "data distribution shift", "n": 3}).json()
for i, q in enumerate(quiz["questions"]):
    print(f"Q{i+1}: {q['question']}")
    for j, o in enumerate(q["options"]):
        print(f"   {j}) {o}")
    print(f"   correct = {q['answer_index']} | best match: page {q['page']}\n")

# 2) Grade one right and one wrong answer through the API
q = quiz["questions"][0]
right = q["options"][q["answer_index"]]
wrong = next(o for j, o in enumerate(q["options"]) if j != q["answer_index"])

for label, ans in [("RIGHT", right), ("WRONG", wrong)]:
    r = requests.post(f"{BASE}/grade",
                      json={"topic": "data distribution shift", "question": q["question"], "answer": ans})
    print(f"--- {label} answer ---")
    print(r.status_code, r.json())

# 3) Check that the scores were saved
print("\nhistory:", requests.get(f"{BASE}/history").json())

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[retry 1/3] bad output, trying again


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q1: What does P(X | Y) represent in the context of machine learning?
   0) Distribution of features given specific labels
   1) Distribution of labels given specific features
   2) Distribution of features
   3) Distribution of labels
   correct = 0 | best match: page 10

Q2: Which type of monitoring focuses on detecting performance issues with label delays?
   0) Fine-grained monitoring
   1) Coarse-grained monitoring
   2) Full-feedback monitoring
   3) Partial-feedback monitoring
   correct = 1 | best match: page 18

Q3: What is a challenge in fine-grained monitoring when dealing with thousands of features?
   0) Computing shift using existing methods works well
   1) Low alert fatigue
   2) Tedious scaling to hundreds of features
   3) No need for DQ verification
   correct = 2 | best match: page 31



[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- RIGHT answer ---
200 {'score': 100, 'feedback': "The student's answer accurately describes what P(X | Y) represents in the context of machine learning.", 'correct_answer': 'Distribution of features given specific labels'}
--- WRONG answer ---
200 {'score': 30, 'feedback': "The student's answer contradicts the context provided. P(X | Y) represents the distribution of features given specific labels, not the distribution of labels given specific features.", 'correct_answer': 'Distribution of features given specific labels'}

history: {'history': [{'topic': 'data distribution shift', 'question': 'What does P(X | Y) represent in the context of machine learning?', 'score': 100, 'time': '2026-10-11 04:40'}, {'topic': 'data distribution shift', 'question': 'What does P(X | Y) represent in the context of machine learning?', 'score': 30, 'time': '2026-10-11 04:40'}], 'average': 65.0, 'weakest_topic': 'data distribution shift'}


In [22]:
!pip install -q streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 28.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 98.4 MB/s eta 0:00:00:00:010:01


In [23]:
%%writefile app.py
import requests
import streamlit as st

BASE = "http://127.0.0.1:8000"  # FastAPI backend running in the notebook
st.set_page_config(page_title="AI Personal Tutor")
st.title("AI Personal Tutor")

tab_ask, tab_quiz, tab_hist = st.tabs(["Ask", "Quiz", "History"])

# ---------- Ask ----------
with tab_ask:
    question = st.text_input("Ask a question about the lecture")
    if st.button("Ask") and question.strip():
        with st.spinner("Thinking..."):
            r = requests.post(f"{BASE}/ask", json={"question": question}, timeout=300)
        st.markdown(r.json()["answer"])

# ---------- Quiz ----------
with tab_quiz:
    topic = st.text_input("Topic", value=st.session_state.get("next_topic", "data distribution shift"))
    n = st.slider("Number of questions", 1, 5, 3)
    if st.button("Generate quiz"):
        with st.spinner("Writing questions..."):
            r = requests.post(f"{BASE}/quiz", json={"topic": topic, "n": n}, timeout=300)
        st.session_state.quiz = r.json()
        st.session_state.quiz_topic = topic  # remember the topic used for grading
        st.session_state.results = {}

    quiz = st.session_state.get("quiz")
    if quiz:
        for i, q in enumerate(quiz["questions"]):
            st.subheader(f"Q{i+1}: {q['question']}")
            choice = st.radio("Your answer", q["options"], key=f"radio_{i}", index=None)
            if st.button("Check", key=f"check_{i}") and choice is not None:
                with st.spinner("Grading..."):
                    r = requests.post(f"{BASE}/grade", json={
                        "topic": st.session_state.quiz_topic,
                        "question": q["question"], "answer": choice}, timeout=300)
                st.session_state.results[i] = r.json()
            res = st.session_state.get("results", {}).get(i)
            if res:
                st.write(f"Score: {res['score']}/100")
                st.write(res["feedback"])
                st.caption(f"Correct answer: {q['options'][q['answer_index']]} "
                           f"(best match: page {q['page']})")

# ---------- History ----------
with tab_hist:
    h = requests.get(f"{BASE}/history", timeout=30).json()
    if not h["history"]:
        st.info("No scores yet.")
    else:
        st.metric("Average score", round(h["average"], 1))
        st.write(f"Weakest topic: **{h['weakest_topic']}**")
        if st.button("Quiz me on my weakest topic"):
            st.session_state.next_topic = h["weakest_topic"]
            st.info("Topic set. Go to the Quiz tab and press Generate quiz.")
        st.dataframe(h["history"])

Writing app.py


In [24]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("NGROK_TOKEN")

In [25]:
import subprocess, time
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

ngrok.kill()  # close old tunnels if the cell is rerun
ngrok.set_auth_token(UserSecretsClient().get_secret("NGROK_TOKEN"))  # Kaggle Secret name

subprocess.Popen(["streamlit", "run", "app.py",
                  "--server.port", "8501", "--server.headless", "true",
                  "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"])
time.sleep(6)
print("Open this link:", ngrok.connect(8501).public_url)

2026-10-11 04:41:09.374 Uvicorn server started on :::8501



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://34.80.111.74:8501

Open this link: https://0cf9-34-80-111-74.ngrok-free.app


In [26]:
# ===== Step 7c-1: /upload endpoint (rebuilds the knowledge base) =====
import os
from fastapi import Request
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi

Chroma = type(vectordb)  # reuse the same Chroma class you used in Step 2

def rebuild_index(pdf_path, name):
    # replace the global objects that dense_search / bm25_search / retrieve use
    global chunks, vectordb, bm25
    pages = PyPDFLoader(pdf_path).load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)
    new_chunks = splitter.split_documents(pages)
    for c in new_chunks:
        c.metadata = {"page": c.metadata["page"] + 1, "source": name}  # 1-based page

    vectordb.delete_collection()
    vectordb = Chroma.from_documents(new_chunks, embeddings, persist_directory="chroma_db")

    # use your own tokenizer from Step 3 if it is named `tokenize`, else simple lowercase split
    tok = globals().get("tokenize", lambda t: t.lower().split())
    bm25 = BM25Okapi([tok(c.page_content) for c in new_chunks])
    chunks = new_chunks
    return len(new_chunks)

# ===== Fix: clear score history automatically when a new PDF is uploaded =====
# NOTE: run this BEFORE the first /upload call. Uvicorn already registered the old route,
# so if you uploaded once already, restart the backend cell (7b) and then run 7c-1 and this cell.
@app.post("/upload")
async def api_upload(request: Request, filename: str = "uploaded.pdf"):
    data = await request.body()
    path = "/kaggle/working/uploaded.pdf"
    with open(path, "wb") as f:
        f.write(data)
    with gpu_lock:
        n = rebuild_index(path, os.path.basename(filename))
    if os.path.exists(SCORES_FILE):
        os.remove(SCORES_FILE)   # old scores belong to the old PDF
    return {"chunks": n, "source": filename}

In [27]:
# ===== Step 7c-2: add an uploader to the Streamlit sidebar =====
extra = '''

# ---------- Upload a new PDF ----------
with st.sidebar:
    st.header("Lecture PDF")
    pdf = st.file_uploader("Upload a PDF", type="pdf", key="pdf_upl")
    if pdf is not None and st.button("Use this PDF", key="use_pdf"):
        with st.spinner("Indexing..."):
            r = requests.post(f"{BASE}/upload", params={"filename": pdf.name},
                              data=pdf.getvalue(), timeout=900)
        st.success(f"Indexed {r.json()['chunks']} chunks from {pdf.name}")
        st.session_state.pop("quiz", None)   # old quiz belongs to the old PDF
'''
src = open("app.py").read()
if "/upload" not in src:          # run-once guard
    open("app.py", "a").write(extra)
print("done")

done


In [28]:
# ===== Fix: /ask returned "" and /upload blocked the server =====
from fastapi import Request
from starlette.concurrency import run_in_threadpool

# 1) Same logic as ask(), but it RETURNS the markdown text
def ask_text(question: str) -> str:
    kind = router_chain.invoke({"question": question})
    if "OVERVIEW" in kind.upper():
        lines = ["## 🗂️ Topics covered in this PDF", ""]
        lines += [f"- **Page {p}** — {t}" for p, t in sorted(outline.items())]
        return "\n".join(lines)
    docs = retrieve(question)
    answer = explain_chain.invoke({"context": format_context(docs), "question": question})
    example = None if "don't know" in answer.lower() else pick_example(question, docs)
    return build_markdown(question, answer, example, docs)

# 2) Rebuild the outline after a new PDF
def rebuild_outline():
    outline.clear()
    for c in chunks:
        title = slide_title(c.page_content)
        if title:
            outline.setdefault(c.metadata["page"], title)

# 3) Remove the old routes, then register the fixed ones
app.router.routes = [r for r in app.router.routes
                     if getattr(r, "path", None) not in ("/ask", "/upload")]

@app.post("/ask")
def api_ask(req: AskReq):
    with gpu_lock:
        return {"answer": ask_text(req.question)}

@app.post("/upload")
async def api_upload(request: Request, filename: str = "uploaded.pdf"):
    data = await request.body()
    path = "/kaggle/working/uploaded.pdf"

    def work():  # heavy part runs in a worker thread, not in the event loop
        with open(path, "wb") as f:
            f.write(data)
        with gpu_lock:
            n = rebuild_index(path, os.path.basename(filename))
            rebuild_outline()
        if os.path.exists(SCORES_FILE):
            os.remove(SCORES_FILE)  # old scores belong to the old PDF
        return n

    n = await run_in_threadpool(work)
    return {"chunks": n, "source": filename}

# ---------- Test ----------
r = requests.post("http://127.0.0.1:8000/ask",
                  json={"question": "What is covariate shift?"}, timeout=300)
print(r.status_code)
print(r.json()["answer"][:300])  # must NOT be empty now

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


200
## 📘 What is covariate shift?

### 💡 Answer
- Covariate shift occurs when the distribution of features \( P(X) \) changes, but the relationship between features and labels \( P(Y | X) \) remains the same. *(best match: page 10)*

### 🧪 Example (a line from the PDF)
- “Covariate shift: More taxi ride


In [29]:
# ===== Fix: grading done by code (exact match), model only writes the feedback =====
from pydantic import BaseModel

class GradeReq2(BaseModel):
    topic: str
    question: str
    answer: str
    correct_answer: str   # sent by the UI from the quiz it already has

# 1) Short feedback chain: the model explains, it does NOT decide the score
fb_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a friendly tutor. Use ONLY the context. In 1-2 sentences, explain "
     "why the correct answer is right. Reply with plain text only."),
    ("human",
     "Context:\n{context}\n\nQuestion: {question}\nCorrect answer: {correct}"),
])
fb_chain = fb_prompt | chat_llm | StrOutputParser()

def grade_choice(question, answer, correct):
    is_right = answer.strip() == correct.strip()      # the code decides
    score = 100 if is_right else 0
    docs = retrieve(question + " " + correct)         # find support for the correct answer
    why = fb_chain.invoke({"context": format_context(docs),
                           "question": question, "correct": correct}).strip()
    head = "Correct." if is_right else f"Not quite. The correct answer is: {correct}."
    return {"score": score, "feedback": f"{head} {why}", "correct_answer": correct}

# 2) Replace the old /grade route
app.router.routes = [r for r in app.router.routes
                     if getattr(r, "path", None) != "/grade"]

@app.post("/grade")
def api_grade(req: GradeReq2):
    with gpu_lock:
        ev = grade_choice(req.question, req.answer, req.correct_answer)
    save_score(req.topic, req.question, ev["score"])
    return ev

In [30]:
src = open("app.py").read()
old = '"question": q["question"], "answer": choice}, timeout=300)'
new = '"question": q["question"], "answer": choice,\n                        "correct_answer": q["options"][q["answer_index"]]}, timeout=300)'
if old in src and "correct_answer" not in src:
    open("app.py", "w").write(src.replace(old, new))
    print("app.py updated")
else:
    print("not changed (already updated, or the text was different)")

app.py updated


Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take 